In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os

DATASET_PATH = "/content/drive/MyDrive/PPG_FieldStudy"

print(os.listdir(DATASET_PATH))

['PPG_FieldStudy_readme.pdf', 'S7', 'S9', 'S6', 'S5', 'S4', 'S2', 'S8', 'S3', 'S15', 'S13', 'S12', 'S14', 'S11', 'S1', 'S10']


In [3]:
import pickle

subject_path = "/content/drive/MyDrive/PPG_FieldStudy/S1/S1.pkl"

with open(subject_path, "rb") as f:
    data = pickle.load(f, encoding="latin1")

print(data.keys())

dict_keys(['rpeaks', 'signal', 'label', 'activity', 'questionnaire', 'subject'])


In [4]:
ppg = data['signal']['wrist']['BVP']
acc = data['signal']['wrist']['ACC']
label = data['label']

print("PPG shape:", ppg.shape)
print("ACC shape:", acc.shape)
print("Label shape:", label.shape)

PPG shape: (589568, 1)
ACC shape: (294784, 3)
Label shape: (4603,)


In [6]:
!pip install scipy numpy pandas matplotlib tqdm scikit-learn

In [7]:
import os
import json
import pickle
import numpy as np

from scipy.signal import butter
from scipy.signal import filtfilt

from tqdm import tqdm

In [8]:
BASE_DIR = "/content/drive/MyDrive/PPG_Project"

PROCESSED_DIR = os.path.join(BASE_DIR, "processed")
SPLIT_DIR = os.path.join(BASE_DIR, "splits")

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(SPLIT_DIR, exist_ok=True)

print("Folders created")

Folders created


In [9]:
# ============================================================
# HEART RATE ESTIMATION PROJECT
# PPG-DaLiA PREPROCESSING
#
# Helper Functions
# ============================================================

def bandpass_filter(
    signal,
    lowcut=0.5,
    highcut=4.0,
    fs=64,
    order=4
):
    """
    Bandpass filter for PPG.

    0.5 Hz  = 30 BPM
    4.0 Hz  = 240 BPM

    Removes:
    - baseline drift
    - motion noise
    - high-frequency artifacts
    """

    nyq = fs / 2

    low = lowcut / nyq
    high = highcut / nyq

    b, a = butter(
        order,
        [low, high],
        btype='band'
    )

    return filtfilt(
        b,
        a,
        signal
    )


def zscore(signal):
    """
    Standard normalization.

    mean = 0
    std = 1
    """

    return (
        signal - np.mean(signal)
    ) / (
        np.std(signal) + 1e-8
    )


def compute_acc_magnitude(acc):
    """
    ACC magnitude

    sqrt(x² + y² + z²)

    Input:
        (N,3)

    Output:
        (N,)
    """

    return np.sqrt(
        acc[:,0]**2 +
        acc[:,1]**2 +
        acc[:,2]**2
    )

In [10]:
# ============================================================
# Create empty lists
#
# Every processed window will be stored here
# ============================================================

all_ppg = []

all_acc_mag = []

all_acc_xyz = []

all_hr = []

all_subjects = []

In [11]:
# ============================================================
# Find all subjects
# ============================================================

subjects = sorted([
    s
    for s in os.listdir(DATASET_PATH)
    if s.startswith("S")
])

print(subjects)

['S1', 'S10', 'S11', 'S12', 'S13', 'S14', 'S15', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9']


In [12]:
# ============================================================
# Process all subjects
# ============================================================

for subject in tqdm(subjects):

    # --------------------------------------------------------
    # Load subject file
    # --------------------------------------------------------

    file_path = os.path.join(
        DATASET_PATH,
        subject,
        f"{subject}.pkl"
    )

    with open(
        file_path,
        "rb"
    ) as f:

        data = pickle.load(
            f,
            encoding="latin1"
        )

    # --------------------------------------------------------
    # Extract signals
    # --------------------------------------------------------

    ppg = data['signal']['wrist']['BVP'].flatten()

    acc = data['signal']['wrist']['ACC']

    label = data['label']

    # --------------------------------------------------------
    # Filter PPG
    # --------------------------------------------------------

    ppg = bandpass_filter(ppg)

    ppg = zscore(ppg)

    # --------------------------------------------------------
    # ACC Magnitude
    # --------------------------------------------------------

    acc_mag = compute_acc_magnitude(acc)

    acc_mag = zscore(acc_mag)

    # --------------------------------------------------------
    # Keep raw xyz channels
    #
    # Future Transformer model
    # can learn motion artifacts
    # --------------------------------------------------------

    acc_xyz = acc.astype(np.float32)

    for k in range(3):

        acc_xyz[:,k] = zscore(
            acc_xyz[:,k]
        )

    # --------------------------------------------------------
    # Windowing
    #
    # PPG:
    # 8 seconds
    # 512 samples
    #
    # ACC:
    # 8 seconds
    # 256 samples
    #
    # Shift:
    # 2 seconds
    # --------------------------------------------------------

    for i in range(len(label)):

        ppg_start = i * 128

        ppg_end = ppg_start + 512

        acc_start = i * 64

        acc_end = acc_start + 256

        if ppg_end > len(ppg):
            break

        if acc_end > len(acc_mag):
            break

        all_ppg.append(
            ppg[
                ppg_start:ppg_end
            ]
        )

        all_acc_mag.append(
            acc_mag[
                acc_start:acc_end
            ]
        )

        all_acc_xyz.append(
            acc_xyz[
                acc_start:acc_end
            ]
        )

        all_hr.append(
            label[i]
        )

        all_subjects.append(
            int(subject[1:])
        )

print("Finished preprocessing")

100%|██████████| 15/15 [06:08<00:00, 24.57s/it]

Finished preprocessing


In [13]:
# ============================================================
# Convert Python lists into NumPy arrays
# ============================================================

X_ppg = np.array(
    all_ppg,
    dtype=np.float32
)

X_acc_mag = np.array(
    all_acc_mag,
    dtype=np.float32
)

X_acc_xyz = np.array(
    all_acc_xyz,
    dtype=np.float32
)

y = np.array(
    all_hr,
    dtype=np.float32
)

subject_ids = np.array(
    all_subjects,
    dtype=np.int32
)

print("X_ppg shape:", X_ppg.shape)

print("X_acc_mag shape:", X_acc_mag.shape)

print("X_acc_xyz shape:", X_acc_xyz.shape)

print("y shape:", y.shape)

print("subject_ids shape:", subject_ids.shape)

X_ppg shape: (64697, 512)
X_acc_mag shape: (64697, 256)
X_acc_xyz shape: (64697, 256, 3)
y shape: (64697,)
subject_ids shape: (64697,)


In [14]:
# ============================================================
# Subject-wise split
#
# Train:
#   S1-S12
#
# Validation:
#   S13
#
# Test:
#   S14-S15
#
# Prevents subject leakage
# ============================================================

train_subjects = list(range(1,13))

val_subjects = [13]

test_subjects = [14,15]

train_idx = np.where(
    np.isin(subject_ids, train_subjects)
)[0]

val_idx = np.where(
    np.isin(subject_ids, val_subjects)
)[0]

test_idx = np.where(
    np.isin(subject_ids, test_subjects)
)[0]

print("Train windows:", len(train_idx))
print("Validation windows:", len(val_idx))
print("Test windows:", len(test_idx))

Train windows: 51690
Validation windows: 4565
Test windows: 8442


In [15]:
# ============================================================
# Save split indices
#
# Every future experiment should use
# the exact same splits.
# ============================================================

np.save(
    f"{SPLIT_DIR}/train_idx.npy",
    train_idx
)

np.save(
    f"{SPLIT_DIR}/val_idx.npy",
    val_idx
)

np.save(
    f"{SPLIT_DIR}/test_idx.npy",
    test_idx
)

print("Split files saved")

Split files saved


In [16]:
# ============================================================
# Dataset statistics
#
# Useful for reports, papers,
# normalization and debugging.
# ============================================================

stats = {

    "num_windows": int(len(y)),

    "mean_hr": float(np.mean(y)),

    "std_hr": float(np.std(y)),

    "min_hr": float(np.min(y)),

    "max_hr": float(np.max(y))
}

print(stats)

{'num_windows': 64697, 'mean_hr': 89.42866516113281, 'std_hr': 22.82712173461914, 'min_hr': 41.69084930419922, 'max_hr': 187.0154266357422}


In [17]:
# ============================================================
# Save statistics as JSON
# ============================================================

with open(
    f"{PROCESSED_DIR}/dataset_stats.json",
    "w"
) as f:

    json.dump(
        stats,
        f,
        indent=4
    )

print("Statistics saved")

Statistics saved


In [18]:
# ============================================================
# Verify everything exists
# ============================================================

print("\nProcessed files:")

for file in os.listdir(PROCESSED_DIR):

    print(file)

print("\nSplit files:")

for file in os.listdir(SPLIT_DIR):

    print(file)


Processed files:
dataset_stats.json

Split files:
train_idx.npy
val_idx.npy
test_idx.npy
